In [1]:
%pip install selenium
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 17.7 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [requests]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import json
import time

HEADERS = {
    "User-Agent": "Sathvik Malla sathvikmalla17@gmail.com",
    "Accept-Encoding": "gzip, deflate",
}

BASE = "https://data.sec.gov"

In [6]:
def get_cik(ticker):
    url = "https://efts.sec.gov/LATEST/search-index?q=%22{}%22&dateRange=custom&startdt=2020-01-01&forms=10-K".format(ticker)

    tickers_url = "https://www.sec.gov/files/company_tickers.json"
    r = requests.get(tickers_url, headers=HEADERS)
    data = r.json()

    for entry in data.values():
        if entry["ticker"].upper() == ticker.upper():
            return str(entry["cik_str"]).zfill(10)
    
    raise ValueError(f"Ticker {ticker} not found")

cik = get_cik("NVDA")
print(cik)

0001045810


In [8]:
def get_filings(cik, form_type="10-K", count=5):
    url = f"{BASE}/submissions/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS)
    data = r.json()

    filings = data["filings"]["recent"]
    results = []

    for i, form in enumerate(filings["form"]):
        if form == form_type:
            results.append({
                "accessionNumber": filings["accessionNumber"][i].replace("-", ""),
                "filingDate": filings["filingDate"][i],
                "form": form,
                "primaryDocument": filings["primaryDocument"][i]
            })
            if len(results) >= count:
                break
    
    return results

In [10]:
def get_filing_text(cik, accession, doc_name):
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{doc_name}"
    r = requests.get(url, headers=HEADERS)
    time.sleep(0.5)  # Respect rate limits
    return r.text

def extract_relationships_from_10k(text, company):
    """
    10K: partnerships, suppliers, customers, subsidiaries, JVs.
    """
    import re
    
    relationships = []
    
    patterns = {
        "partnership": r"(?:partnership|partnered|strategic alliance).*?(?:with|between)\s+([A-Z][A-Za-z\s,]+(?:Inc|Corp|LLC|Ltd|Co\.)?)",
        "supplier": r"([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)\s+(?:supplies|is (?:a|our) (?:key |primary )?supplier)",
        "customer": r"([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)\s+(?:is|was|are|were)\s+(?:a|our|the)\s+(?:significant|major|key|largest)\s+customer",
        "acquisition": r"(?:acquired|acquisition of|merger with)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "investment": r"(?:invested in|investment in|equity stake in)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
        "subsidiary": r"(?:wholly.owned subsidiary|our subsidiary)\s+([A-Z][A-Za-z\s]+(?:Inc|Corp|LLC)?)",
    }
    
    for rel_type, pattern in patterns.items():
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches[:10]:
            relationships.append({
                "source": company,
                "target": match.strip(),
                "type": rel_type,
                "filing": "10-K"
            })
    
    return relationships

In [ ]:
def get_ownership_filings(cik: str) -> list:
    results = []
    
    for form in ["SC 13D", "SC 13G"]:
        filings = get_filings(cik, form_type=form, count=10)
        for f in filings:
            results.append({
                "form": form,
                "date": f["filingDate"],
                "accession": f["accessionNumber"],
                "influence_type": "activist" if "13D" in form else "passive",
                "target_company_cik": cik,
            })
    
    return results

def parse_13d_for_investor(cik: str) -> dict:
    filings = get_filings(cik, form_type="SC 13D", count=3)
    ownership_data = []
    
    for filing in filings:
        url = f"{BASE}/submissions/CIK{cik}.json"
        # Parse filer name, ownership %, purpose of transaction
        # from the filing document text
        ownership_data.append({
            "filing_date": filing["filingDate"],
            "accession": filing["accessionNumber"],
        })
    
    return ownership_data